In [7]:
import os, io, time, shutil, glob, pathlib, json, requests
import pandas as pd
import polars as pl
from datetime import datetime, timedelta
from PIL import Image
import win32clipboard
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoAlertPresentException
from urllib.parse import urlparse

TEAMS_WEBHOOK_URL = "https://default599e51d62f8c43478e591f795a51a9.8c.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/c24f30c010df45a6a6dac9421643bb34/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=5vWDl18a7-IWSvHuZAWgGtQcwM54nEapSArj4JVPnGg"

DATA_DIR      = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_agent"
CURRENT_INTERVAL_DIR = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_interval"

EXPEDIA_URL   = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentRealtime"

OPTICS_INTERVAL_URL = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentBreakdownRealtimeDashboard"

CHROMEDRIVER  = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\chromedriver-win64\chromedriver.exe"
SOURCE_FOLDER = r"C:\temp\expedia_downloads"

LOB_MAP = {
    "LG Chat": [
        "Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging",
        "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car",
        "Chat_AC_GLB_EN_Lodging_Proficient", "Chat_AC_GLB_EN_Car_Activity",
        "Chat_AC_GLB_EN_Lodging_Expert",
    ],
    "NL Chat": [
        "Chat - Global English Non- Lodging Nesting", "Chat_OD_EN_Dual_GDS",
        "Chat_AC_GLB_EN_Proficient", "Chat_AC_GLB_EN_Expert",
    ],
}

LOC_STYLE = {
    "HCM":   {"bg": "#DA251D", "fg": "#FFD700"},
    "PUN":   {"bg": "#388E3C", "fg": "#ffffff"},
    "KOL":   {"bg": "#1565C0", "fg": "#ffffff"},
    "CAI":   {"bg": "#E65100", "fg": "#ffffff"},
    "OTHER": {"bg": "#757575", "fg": "#ffffff"},
}
LOB_STYLE = {
    "LG Chat": {"bg": "#1565C0", "fg": "#ffffff"},
    "NL Chat": {"bg": "#2E7D32", "fg": "#ffffff"},
}
NOTE_STYLE = {
    "⚠️ over-break":      {"bg": "#E65100", "fg": "#ffffff"},
    "⚠️ over-lunch":      {"bg": "#B71C1C", "fg": "#ffffff"},
    "⚠️ available idle":  {"bg": "#F57F17", "fg": "#ffffff"},
    "⚠️ offline w/ work": {"bg": "#880E4F", "fg": "#ffffff"},
    "⚠️ unproductive":    {"bg": "#37474F", "fg": "#ffffff"},
    "🔍 need to check":   {"bg": "#4A148C", "fg": "#ffffff"},
    "OK":                 {"bg": "#E8F5E9", "fg": "#2E7D32"},
}
CONNECT_STATE_STYLE = {
    "AVAILABLE":    ("#E3F2FD", "#1565C0"),
    "READY":        ("#E3F2FD", "#1565C0"),
    "BREAK":        ("#FFE0B2", "#BF360C"),
    "LUNCH":        ("#C8E6C9", "#1B5E20"),
    "COACHING":     ("#EDE7F6", "#4A148C"),
    "TRAINING":     ("#E8EAF6", "#283593"),
    "TEAM MEETING": ("#E8EAF6", "#283593"),
    "OFFLINEWORK":  ("#FFCCBC", "#BF360C"),
    "NOT READY":    ("#FFEBEE", "#C62828"),
    "NOTREADY":     ("#FFEBEE", "#C62828"),
    "UNAVAILABLE":  ("#FFEBEE", "#C62828"),
    "ENDOFSHIFT":   ("#FAFAFA", "#757575"),
    "LOGIN":        ("#FFF8E1", "#F57F17"),
    "PERSONAL":     ("#FBE9E7", "#BF360C"),
}

In [8]:
def convert_to_datetime(st):
    return datetime(*st[:6])

def input_data(data_dir):
    files = []
    for fn in pathlib.Path(data_dir).glob('**/*.*'):
        sfx = fn.suffixes
        if not (sfx and sfx[-1].lower() in ['.xlsx', '.csv']): continue
        exp_dt = convert_to_datetime(time.localtime(os.path.getmtime(fn)))
        try:
            if sfx[-1].lower() == '.xlsx':
                df = pl.read_excel(fn)
            else:
                if os.path.getsize(fn) == 0: continue
                df = pl.read_csv(fn, infer_schema_length=10000)
            if df.is_empty(): continue
            df = df.with_columns([pl.lit(fn.stem).alias('sheet_name'), pl.lit(exp_dt).alias('Export time')])
            files.append(df)
        except Exception as e:
            print(f"❌ {fn.name}: {e}")
    return pl.concat(files, how='diagonal_relaxed') if files else pl.DataFrame()

def download_current_interval(driver):
    check_and_login(driver, OPTICS_INTERVAL_URL)
    wait=WebDriverWait(driver,15)
    try:
        btns=wait.until(lambda d: d.find_elements(By.CSS_SELECTOR,"button.settingsButton"))
        if not btns: raise TimeoutException("Không tìm thấy settingsButton")
        btn=btns[0]
        driver.execute_script("arguments[0].scrollIntoView(true);",btn); time.sleep(0.5)
        driver.execute_script("arguments[0].click();",btn)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,"div.uitk-menu-container[aria-hidden='false']")))
        wait.until(EC.element_to_be_clickable((By.XPATH,
            "//div[contains(@class,'uitk-menu-container') and contains(@class,'uitk-menu-open')]"
            "[@aria-hidden='false']//button[contains(@class,'uitk-list-item')]"
            "//span[text()='Download CSV']/ancestor::button"))).click()
        print("✅ Downloaded Current Interval CSV"); time.sleep(5)
    except Exception as e:
        print(f"❌ download_current_interval: {e}"); return
    os.makedirs(CURRENT_INTERVAL_DIR,exist_ok=True)
    for f in pathlib.Path(SOURCE_FOLDER).glob("Current Interval*"):
        if f.suffix.lower() not in ('.csv','.xlsx') or f.name.endswith('.crdownload'): continue
        dst=pathlib.Path(CURRENT_INTERVAL_DIR)/f.name
        if dst.exists(): dst.unlink()
        shutil.move(str(f),str(dst))
        print(f"📁 Moved: {f.name}")

def load_current_interval():
    files=[]
    for fn in pathlib.Path(CURRENT_INTERVAL_DIR).glob("Current Interval*"):
        if fn.suffix.lower() not in ('.csv','.xlsx'): continue
        try:
            exp_dt=convert_to_datetime(time.localtime(os.path.getmtime(fn)))
            df=pl.read_csv(fn,infer_schema_length=10000) if fn.suffix.lower()=='.csv' \
               else pl.read_excel(fn,engine='calamine')
            if df.is_empty(): continue
            df=df.with_columns(pl.lit(exp_dt).alias('Export time'))
            files.append(df)
        except: continue
    if not files: print("⚠️ Không có file Current Interval"); return pd.DataFrame()
    combined=pl.concat(files,how='diagonal_relaxed')
    latest=combined['Export time'].max()
    return combined.filter(pl.col('Export time')==latest).drop('Export time').to_pandas()

def check_and_login(driver, url):
    driver.get(url); time.sleep(10)
    try:
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, 'button[data-testid="console-okta-sign-in"]'))).click()
        print("🔑 Signing in..."); time.sleep(2)
        try:
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'label[for="input36"][data-se-for-name="rememberMe"]'))).click()
            time.sleep(1)
        except TimeoutException: pass
        try:
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input.button.button-primary[type="submit"][value="Next"]'))).click()
            time.sleep(10)
        except TimeoutException: pass
        print("🎉 Login successful!")
        try: driver.switch_to.alert.accept()
        except NoAlertPresentException: pass
        driver.get(url)
    except TimeoutException:
        print("✅ No sign-in required.")

def str_hms_to_seconds(hms):
    try:
        p = [int(x) for x in str(hms).split(':')]
        if len(p)==3: return p[0]*3600+p[1]*60+p[2]
        if len(p)==2: return p[0]*60+p[1]
        return int(p[0])
    except: return None

def process_outage(df, select_cols):
    df = df.sort(["LOB","Connect State","Duration (s)"], descending=[False,False,True])
    total = df.shape[0]
    avail = [c for c in select_cols if c in df.columns]
    return df.head(50).select(avail).to_pandas(), total

def seconds_to_hms(s):
    try:
        s=int(float(s)); h,r=divmod(s,3600); m,sec=divmod(r,60)
        return f'{h:02d}:{m:02d}:{sec:02d}'
    except: return '—'

def hms_to_seconds(hms):
    try:
        p=str(hms).split(':'); return int(p[0])*3600+int(p[1])*60+int(p[2])
    except: return 0

def get_duration_color(hms_str):
    try:
        s=hms_to_seconds(hms_str)
        if s>3600: return "#b71c1c","#ffffff"
        if s>1800: return "#e53935","#ffffff"
        if s> 900: return "#fb8c00","#ffffff"
        if s> 300: return "#fdd835","#1a1a1a"
    except: pass
    return None, None

def make_bar_cell(s, max_s, color):
    f=int((s/max_s if max_s>0 else 0)*150); e=150-f
    bg=color if color else "#43a047"
    return (
        '<td style="padding:3px 8px;border:1px solid #ddd;vertical-align:middle;" nowrap>'
        '<table cellpadding="0" cellspacing="0" border="0" width="150" style="border-collapse:collapse;"><tr height="14">'
        f'<td width="{f}" height="14" bgcolor="{bg}" style="height:14px;font-size:10px;line-height:14px;">{"&nbsp;"if f>0 else""}</td>'
        f'<td width="{e}" height="14" bgcolor="#e0e0e0" style="height:14px;font-size:10px;line-height:14px;">{"&nbsp;"if e>0 else""}</td>'
        '</tr></table></td>'
    )

def build_html_table(df, title, is_global=False, cases=0, summary=''):
    now=datetime.now()-(timedelta(hours=14) if is_global else timedelta(0))
    tz='(PST)' if is_global else '(VNT)'
    sub=f"Updated {now.strftime('%d-%b-%Y')} · {now.strftime('%I:%M %p')} {tz}"
    df=df.copy().drop(columns=[c for c in ['sheet_name','Export time'] if c in df.columns])
    if 'Duration (s)' in df.columns:
        df['Duration']=df['Duration (s)'].apply(seconds_to_hms)
        df=df.drop(columns=['Duration (s)'])
    cols=list(df.columns)
    dur_idx   = cols.index('Duration')      if 'Duration'      in cols else -1
    lob_idx   = cols.index('LOB')           if 'LOB'           in cols else -1
    note_idx  = cols.index('Note')          if 'Note'          in cols else -1
    state_idx = cols.index('Connect State') if 'Connect State' in cols else -1
    loc_idx   = cols.index('Location')      if 'Location'      in cols else -1
    staff_att_idx = cols.index('Staff Attainment') if 'Staff Attainment' in cols else -1
    agent_def_idx = cols.index('Agent Deficit')    if 'Agent Deficit'    in cols else -1
    dur_secs  = [hms_to_seconds(str(row[cols[dur_idx]])) for _,row in df.iterrows()] if dur_idx>=0 else [0]*len(df)
    max_s     = max(dur_secs) if dur_secs and dur_idx>=0 else 1
    th='bgcolor="#1e3a5f" style="color:#ffffff;padding:7px 12px;border:1px solid #2c4f7c;text-align:left;" nowrap'
    hdrs=[f'<th {th}>{c}</th>' for c in cols]
    if dur_idx>=0: hdrs.insert(dur_idx+1, f'<th {th}>Duration Bar</th>')
    def tdp(bg,v): return f'<td bgcolor="{bg}" style="color:#1a1a1a;padding:6px 12px;border:1px solid #ddd;" nowrap>{v}</td>'
    def tdb(bg,fg,v,al='left'): return f'<td bgcolor="{bg}" style="color:{fg};padding:6px 12px;border:1px solid #ddd;font-weight:bold;text-align:{al};" nowrap>{v}</td>'
    rows_html=''
    for i,(_,row) in enumerate(df.iterrows()):
        rbg='#f0f4ff' if i%2==0 else '#ffffff'
        lv=str(row[cols[lob_idx]])  if lob_idx >=0 else ''
        nv=str(row[cols[note_idx]]) if note_idx>=0 else ''
        cells=[]
        for j,col in enumerate(cols):
            v=str(row[col]) if pd.notna(row[col]) else '—'
            if j==dur_idx:
                bg,fg=get_duration_color(v); cells.append(tdb(bg,fg,v) if bg else tdp(rbg,v))
                cells.append(make_bar_cell(dur_secs[i],max_s,bg))
            elif j==loc_idx:
                s=LOC_STYLE.get(v); cells.append(tdb(s['bg'],s['fg'],v,'center') if s else tdp(rbg,v))
            elif j==lob_idx:
                s=LOB_STYLE.get(lv); cells.append(tdb(s['bg'],s['fg'],v,'center') if s else tdp(rbg,v))
            elif j==note_idx:
                s=NOTE_STYLE.get(nv,{'bg':rbg,'fg':'#1a1a1a'}); cells.append(tdb(s['bg'],s['fg'],v,'center'))
            elif j==state_idx:
                sc=CONNECT_STATE_STYLE.get(v); cells.append(tdb(sc[0],sc[1],v) if sc else tdp(rbg,v))
            elif j==staff_att_idx:
                try:
                    n=float(v.replace('%','').replace(',',''))
                    bg='#1B5E20' if n>=95 else '#B71C1C'
                    cells.append(tdb(bg,'#ffffff',v,'center'))
                except: cells.append(tdp(rbg,v))
            elif j==agent_def_idx:
                try:
                    n=float(v.replace(',',''))
                    bg='#1B5E20' if n<=0 else '#B71C1C'
                    cells.append(tdb(bg,'#ffffff',v,'center'))
                except: cells.append(tdp(rbg,v))
            else:
                cells.append(tdp(rbg,v))
        rows_html+=f"<tr>{''.join(cells)}</tr>"
    sh=f'  <span style="font-size:11px;">📊 {summary}</span><br>\n' if summary else ''
    return (f'<p>\n  <b style="color:#c0392b;font-size:16px;">🔴 {title}</b><br>\n'
            f'  <span style="font-size:12px;">{sub} &nbsp;|&nbsp; ⚡ <b>{cases} CASES</b></span><br>\n{sh}</p>\n'
            f'<div style="overflow-x:auto;">\n<table border="1" cellpadding="0" cellspacing="0" '
            f'style="border-collapse:collapse;font-size:12px;font-family:Segoe UI,Arial,sans-serif;">\n'
            f'  <thead><tr>{"".join(hdrs)}</tr></thead>\n  <tbody>{rows_html}</tbody>\n</table>\n</div>')

def send_html_via_webhook(df, title, is_global=False, cases=0, summary=''):
    if df is None or (hasattr(df,'empty') and df.empty):
        print(f"⏭️  Skipping '{title}'"); return
    payload={'html': build_html_table(df, title, is_global, cases, summary)}
    try:
        r=requests.post(TEAMS_WEBHOOK_URL, headers={'Content-Type':'application/json'},
                        data=json.dumps(payload), timeout=30)
        print(f"✅ Sent: '{title}'" if r.status_code in (200,202) else f"❌ Failed [{r.status_code}]: {r.text[:200]}")
    except Exception as e:
        print(f"❌ {e}")

In [9]:
start_time    = datetime.now()
path_fragment = urlparse(EXPEDIA_URL).path.split('/')[-1]
chrome_options= Options()
chrome_options.add_argument(r'--user-data-dir=C:/temp/new_chrome_profile')
chrome_options.add_argument(r'--profile-directory=Default')
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=chrome_options)

check_and_login(driver, EXPEDIA_URL)
wait = WebDriverWait(driver, 10)
try:
    btn = wait.until(lambda d: d.execute_script("""
        const el=Array.from(document.querySelectorAll('*')).find(e=>
            e.childNodes.length===1&&e.childNodes[0].nodeType===Node.TEXT_NODE&&
            e.textContent.trim()==='Logged-In Agents');
        if(!el)return null;
        let n=el.parentElement;
        while(n&&n!==document.body){const b=n.querySelectorAll('button.settingsButton');if(b.length===1)return b[0];n=n.parentElement;}
        return null;"""))
    if btn is None: raise Exception("settingsButton not found")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});",btn); time.sleep(0.5)
    driver.execute_script("arguments[0].click();",btn)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,"div.uitk-menu-container[aria-hidden='false']")))
    wait.until(EC.element_to_be_clickable((By.XPATH,
        "//div[contains(@class,'uitk-menu-open')][@aria-hidden='false']//span[text()='Download CSV']/ancestor::button"))).click()
    print("✅ Downloaded Logged-In Agents CSV")
except Exception as e:
    print(f"❌ {e}")
wait.until(EC.url_contains(path_fragment)); time.sleep(10)
download_current_interval(driver)
driver.quit()

for pat in [f"{SOURCE_FOLDER}\\Logged-In Agents*.csv", f"{SOURCE_FOLDER}\\Logged-In Agents*.xlsx"]:
    for fp in glob.glob(pat):
        dst=os.path.join(DATA_DIR, os.path.basename(fp)); shutil.move(fp,dst); print(f"Moved: {os.path.basename(fp)}")

outage_db = input_data(DATA_DIR)
if outage_db.is_empty(): raise RuntimeError("❌ No data loaded")
outage_db = outage_db.sort(["Export time"]).filter(pl.col("Export time")==pl.col("Export time").max())

lob_expr = pl.lit(None).cast(pl.Utf8)
for lbl,qs in LOB_MAP.items():
    lob_expr=pl.when(pl.col("Queue Group / Routing Profile").is_in(qs)).then(pl.lit(lbl)).otherwise(lob_expr)
outage_db=(outage_db
    .with_columns(lob_expr.alias("LOB"))
    .filter(pl.col("LOB").is_in(["NL Chat","LG Chat"]))
    .with_columns(
        pl.when(pl.col("Business Location").str.contains("Ho Chi Minh")).then(pl.lit("HCM"))
        .when(pl.col("Business Location").str.contains("Pune")).then(pl.lit("PUN"))
        .when(pl.col("Business Location").str.contains("Kolkata")).then(pl.lit("KOL"))
        .when(pl.col("Business Location").str.contains("Cairo")).then(pl.lit("CAI"))
        .otherwise(pl.lit("OTHER")).alias("Location"))
    .with_columns(pl.col("Duration").cast(str).map_elements(str_hms_to_seconds,return_dtype=pl.Int64).alias("Duration (s)")))
print(f"✅ {len(outage_db)} rows | {outage_db['Location'].unique().to_list()}")

bl_out = outage_db.filter(pl.col("Connect State").is_in(["BREAK","LUNCH"])).with_columns(
    pl.when((pl.col("Connect State")=="BREAK")&(pl.col("Duration (s)")>900)).then(pl.lit("⚠️ over-break"))
    .when((pl.col("Connect State")=="LUNCH")&(pl.col("Duration (s)")>3600)).then(pl.lit("⚠️ over-lunch"))
    .otherwise(pl.lit("OK")).alias("Note"))
over_out = bl_out.filter(pl.col("Note")!="OK")

ct_out = outage_db.filter(~pl.col("Connect State").is_in(["BREAK","LUNCH","AVAILABLE","READY","OFFLINEWORK"])).with_columns(
    pl.when(pl.col("Connect State").is_in(["COACHING","TRAINING","TEAM MEETING"]))
    .then(pl.lit("🔍 need to check")).otherwise(pl.lit("⚠️ unproductive")).alias("Note"))

ac_out = outage_db.filter(pl.col("Connect State").is_in(["AVAILABLE","READY","OFFLINEWORK"])).with_columns(
    pl.when(pl.col("Connect State").is_in(["AVAILABLE","READY"])&
            (pl.col("Assigned Workitem Count").is_null()|(pl.col("Assigned Workitem Count")==0))&
            (pl.col("Duration (s)")>1800)).then(pl.lit("⚠️ available idle"))
    .when((pl.col("Connect State")=="OFFLINEWORK")&(pl.col("Assigned Workitem Count")>=1)&
          (pl.col("Duration (s)")>600)).then(pl.lit("⚠️ offline w/ work"))
    .otherwise(pl.lit("OK")).alias("Note"))

cat_db = outage_db.with_columns(
    pl.when((pl.col("Assigned Workitem Count")>=1)|pl.col("Connect State").is_in(["AVAILABLE","READY"])).then(pl.lit("Available"))
    .when(pl.col("Connect State")=="BREAK").then(pl.lit("Break-Idle"))
    .when(pl.col("Connect State")=="LUNCH").then(pl.lit("Lunch-Idle"))
    .when(pl.col("Connect State").is_in(["COACHING","TEAM MEETING"])).then(pl.lit("Coaching-Idle"))
    .when(pl.col("Connect State")=="TRAINING").then(pl.lit("Training-Idle"))
    .otherwise(pl.lit("Other")).alias("Category"))
pivot_g = cat_db.group_by(["Location","LOB","Category"]).agg(pl.col("Agent Name").n_unique().alias("Count")).pivot(values="Count",index=["Location","LOB"],columns="Category").fill_null(0)
for c in ["Available","Break-Idle","Lunch-Idle","Coaching-Idle","Training-Idle","Other"]:
    if c not in pivot_g.columns: pivot_g=pivot_g.with_columns(pl.lit(0).cast(pl.Int64).alias(c))
pivot_g = pivot_g.select(["Location","LOB","Available","Break-Idle","Lunch-Idle","Coaching-Idle","Training-Idle","Other"]).sort(["Location","LOB"])

def _bl_str(df):
    if df.shape[0]==0: return "No cases"
    cnt=df.group_by(["LOB","Connect State"]).agg(pl.len().alias("Count")).sort(["LOB","Connect State"])
    p={}
    for r in cnt.iter_rows(named=True): p.setdefault(r["LOB"],[]).append(f"{r['Connect State']} ×{r['Count']}")
    return "  |  ".join(f"<b>{k}</b>: {', '.join(v)}" for k,v in sorted(p.items()))

B=["Location","Agent Name","Agent Manager","Connect State","Duration (s)","LOB","Note"]
bl_pd,  bl_n  = process_outage(bl_out,  B)
ov_pd,  ov_n  = process_outage(over_out,B)
ct_pd,  ct_n  = process_outage(ct_out,  B)
ac_pd,  ac_n  = process_outage(ac_out,  ["Location","Agent Name","Agent Manager","Connect State","Assigned Workitem Count","Duration (s)","LOB","Note"])
pivot_pd = pivot_g.to_pandas()

print(f"IC:{len(pivot_pd)} | BL:{bl_n} | Over:{ov_n} | CT:{ct_n} | AC:{ac_n}")

ci_pd = load_current_interval()
send_html_via_webhook(ci_pd,    "Current Interval — Agent Breakdown", cases=len(ci_pd))
send_html_via_webhook(pivot_pd, 'IC Overall — All Sites',             cases=len(pivot_pd))
send_html_via_webhook(bl_pd,    'Lunch / Break',                      cases=bl_n,  summary=_bl_str(bl_out))
send_html_via_webhook(ov_pd,    'Overbreak / Overlunch',              cases=ov_n,  summary=_bl_str(over_out))
send_html_via_webhook(ct_pd,    'Coaching / Training / Unproductive', cases=ct_n)
send_html_via_webhook(ac_pd,    'Available-IDLE',                     cases=ac_n)

✅ No sign-in required.
✅ Downloaded Logged-In Agents CSV
✅ No sign-in required.
✅ Downloaded Current Interval CSV
📁 Moved: Current Interval-Thu Jul 30 2026 12_45_45 GMT+0700 (Indochina Time).csv
Moved: Logged-In Agents-Thu Jul 30 2026 12_45_14 GMT+0700 (Indochina Time).csv
✅ 72 rows | ['KOL', 'HCM', 'CAI']
IC:5 | BL:13 | Over:1 | CT:4 | AC:55


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_21564\4264105125.py:81: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  pivot_g = cat_db.group_by(["Location","LOB","Category"]).agg(pl.col("Agent Name").n_unique().alias("Count")).pivot(values="Count",index=["Location","LOB"],columns="Category").fill_null(0)


✅ Sent: 'Current Interval — Agent Breakdown'
✅ Sent: 'IC Overall — All Sites'
✅ Sent: 'Lunch / Break'
✅ Sent: 'Overbreak / Overlunch'
✅ Sent: 'Coaching / Training / Unproductive'
✅ Sent: 'Available-IDLE'
